# Sprint 7E Target-Observation Context Feature Subgroup Ablation Runner

Colab is a runner only. Model, masking, metrics, diagnostics, figures, and reports are implemented in repository code.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
set -euo pipefail
REPO_URL="${REPO_URL:-https://github.com/YasinEkici/crispr-gnn-offtarget.git}"
BRANCH="${BRANCH:-sprint7/gat-gatv2}"
cd /content
if [ ! -d crispr-gnn-offtarget/.git ]; then
  git clone --branch "$BRANCH" "$REPO_URL" crispr-gnn-offtarget
else
  cd crispr-gnn-offtarget
  git fetch origin "$BRANCH"
  git checkout "$BRANCH"
  git pull --ff-only origin "$BRANCH"
fi
cd /content/crispr-gnn-offtarget
git status --short --branch

In [ ]:
%%bash
set -euo pipefail
cd /content/crispr-gnn-offtarget
python -m pip install -q uv
uv sync

In [ ]:
%%bash
set -euo pipefail
cd /content/crispr-gnn-offtarget
DRIVE_ROOT="${DRIVE_ROOT:-/content/drive/MyDrive/crispr_gnn_offtarget}"
ALT_DRIVE_ROOT="/content/drive/MyDrive/crispr-gnn-offtarget"
if [ ! -d "$DRIVE_ROOT" ] && [ -d "$ALT_DRIVE_ROOT" ]; then
  DRIVE_ROOT="$ALT_DRIVE_ROOT"
fi
echo "Using DRIVE_ROOT=$DRIVE_ROOT"
test -d "$DRIVE_ROOT"
mkdir -p data/raw data/processed
if [ -d "$DRIVE_ROOT/data/raw" ]; then
  rsync -a "$DRIVE_ROOT/data/raw/" data/raw/
fi
if [ -d "$DRIVE_ROOT/data/processed" ]; then
  rsync -a "$DRIVE_ROOT/data/processed/" data/processed/
fi
if [ ! -d data/processed/graphs/sprint5b/graph_c_context_observation ]; then
  echo "Building Sprint 5B Graph C S5F2 artifact required by Sprint 7E..."
  uv run python scripts/build_sprint5b_graph_c_energy_features.py
fi
test -d data/processed/graphs/sprint5b/graph_c_context_observation
find data/processed/graphs -maxdepth 3 -type f -name 'manifest.json' | sort

In [ ]:
%%bash
set -euo pipefail
cd /content/crispr-gnn-offtarget
uv run python scripts/analyze_sprint7e_target_context_features.py

In [ ]:
%%bash
set -euo pipefail
cd /content/crispr-gnn-offtarget
RUN_ID="sprint7e_target_context_subgroup_seed42_$(date -u +%Y%m%d_%H%M%S)"
uv run python scripts/run_sprint7e_target_context_subgroup_ablation.py \
  --config configs/sweeps/sprint7e_target_context_subgroup_ablation.yaml \
  --run-id "$RUN_ID"
echo "$RUN_ID" > outputs/sprint7e/latest_run_id.txt

In [ ]:
%%bash
set -euo pipefail
cd /content/crispr-gnn-offtarget
test -f outputs/sprint7e/context_feature_profiling/sprint7e_context_feature_family_map.csv
test -f outputs/sprint7e/context_feature_profiling/sprint7e_context_feature_profile_report.md
test -f outputs/sprint7e/target_context_subgroup_ablation.csv
test -f outputs/sprint7e/target_context_subgroup_ablation_report.md
test -f outputs/sprint7e/target_context_subgroup_ablation_run_manifest.json
test -f outputs/sprint7e/diagnostics/target_context_subgroup_mask_audit.csv
test -f outputs/sprint7e/figures/target_context_subgroup_auprc_comparison.png
test -f outputs/sprint7e/figures/target_context_subgroup_attention_by_edge_kind.png
uv run python - <<'PY'
import pandas as pd
results = pd.read_csv('outputs/sprint7e/target_context_subgroup_ablation.csv')
expected = {
    'S7E_R1_mask_target_sequence',
    'S7E_R2_mask_experimental_epigenetic',
    'S7E_R3_mask_computed_nucleosome_aggregates',
    'S7E_R4_mask_computed_nucleosome_missingness',
    'S7E_R5_mask_all_nonsequence_context',
}
missing = expected - set(results['predeclared_run_id'].astype(str))
if missing:
    raise SystemExit(f'missing Sprint 7E run ids: {sorted(missing)}')
audit = pd.read_csv('outputs/sprint7e/diagnostics/target_context_subgroup_mask_audit.csv')
if set(audit['context_edges_used']) != {0}:
    raise SystemExit('Sprint 7E canonical runs must drop context edges')
if audit['candidate_attention_attr_abs_sum'].min() <= 0:
    raise SystemExit('candidate S5F2 attention attrs must remain active')
print(results[['predeclared_run_id','test_auprc','test_mcc','test_specificity','test_tn','test_fp']].to_string(index=False))
PY

In [ ]:
%%bash
set -euo pipefail
cd /content/crispr-gnn-offtarget
DRIVE_ROOT="${DRIVE_ROOT:-/content/drive/MyDrive/crispr_gnn_offtarget}"
ALT_DRIVE_ROOT="/content/drive/MyDrive/crispr-gnn-offtarget"
if [ ! -d "$DRIVE_ROOT" ] && [ -d "$ALT_DRIVE_ROOT" ]; then
  DRIVE_ROOT="$ALT_DRIVE_ROOT"
fi
RUN_ID="$(cat outputs/sprint7e/latest_run_id.txt)"
DEST="$DRIVE_ROOT/returned_outputs/$RUN_ID"
mkdir -p "$DEST"
rsync -a --exclude='model.pt' --exclude='.DS_Store' outputs/sprint7e/ "$DEST/"
echo "Copied Sprint 7E outputs to $DEST"